# 读出验证与数值修准

重新拟合原 254 场景的读出器，测试 1007 场景，并在 304 场景修准净力矩；72 场景包含随机方向对照。

先运行 Gemma-12B，再新建运行时将 MODEL_KEY 改为 qwen9b。层与 ridge alpha 固定，使用 BF16、batch size 1；建议 A100 80GB。

[实验设置与运行说明](../../../docs/experiment_notes/03_section44_readout_correction/7ba0519967bb0c4d.md)


In [ ]:
# Run once. Every later code cell reports both cell and inner-loop progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready.')

IMPLEMENTATION_SHA256 = '160372af558929c42134517d3ffc09527278d018135dd39a46f32eb6987dde7d'
IMPLEMENTATION_SHA256 = '3dd569a9b6b0eeddbe0095a7a54ced6dd138010f229d8736918b3611aac8afce'


In [ ]:
%%cell_progress 1/17 Mount Drive
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque project')
from pathlib import Path
PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (
        path for path in PROJECT_CANDIDATES
        if (path / 'toolkit' / 'pyproject.toml').is_file()
    ),
    None,
)
assert PROJECT_ROOT is not None, (
    'Could not find VisualTorque/toolkit/pyproject.toml on Drive.'
)
TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
print('Project:', PROJECT_ROOT)


In [ ]:
%%cell_progress 2/17 Install inference environment
import importlib
import os

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import subprocess
import sys

cell_stage('Removing unused TorchAudio')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'],
    check=False,
)
cell_stage('Installing pinned inference and analysis dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers==5.14.0', 'accelerate>=1.10', 'sentencepiece',
    'protobuf', 'huggingface_hub>=0.34', 'hf_xet>=1.1', 'tqdm>=4.66',
    'pandas>=2.2', 'matplotlib>=3.8', 'scikit-learn>=1.5',
])

cell_stage('Registering the live VisualTorque source tree')
SRC_ROOT = TOOLKIT_ROOT / 'src'
PACKAGE_ROOT = SRC_ROOT / 'visual_torque_toolkit'
assert (PACKAGE_ROOT / '__init__.py').is_file(), PACKAGE_ROOT
source_text = str(SRC_ROOT)
sys.path[:] = [entry for entry in sys.path if entry != source_text]
sys.path.insert(0, source_text)
for module_name in tuple(sys.modules):
    if (
        module_name == 'visual_torque_toolkit'
        or module_name.startswith('visual_torque_toolkit.')
    ):
        del sys.modules[module_name]
importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
import visual_torque_toolkit
from IPython.display import display
from transformers import AutoModelForMultimodalLM, AutoProcessor
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter

required_features = (
    '_template', '_to_device', '_candidate_plan', '_append_text_prefix',
)
assert all(
    hasattr(HuggingFaceMultimodalAdapter, name)
    for name in required_features
)
toolkit_path = Path(visual_torque_toolkit.__file__).resolve()
assert PACKAGE_ROOT.resolve() in toolkit_path.parents, toolkit_path
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('torchvision:', torchvision.__version__)
print('Toolkit source:', toolkit_path)


In [ ]:
%%cell_progress 3/17 Check GPU access
cell_stage('Checking CUDA device')
assert torch.cuda.is_available(), 'Select a GPU runtime with enough memory for the pinned model.'
print(torch.cuda.get_device_name(0))
print('CUDA memory (GB):', round(
    torch.cuda.get_device_properties(0).total_memory / 2**30, 1
))

cell_stage('Authenticating with Hugging Face')
from google.colab import userdata
from huggingface_hub import login
try:
    token = userdata.get('HF_TOKEN')
    if token:
        login(token=token, add_to_git_credential=False)
        print('Hugging Face login complete.')
    else:
        print('HF_TOKEN is not set. Public checkpoints may still work.')
except Exception as exc:
    print('Hugging Face login skipped:', exc)


In [ ]:
%%cell_progress 4/17 Load shared validation utilities
"""Pure CPU utilities embedded in the follow-up notebooks; no model execution."""
import hashlib
import json
from pathlib import Path
from collections import defaultdict, Counter
from itertools import combinations, product
import numpy as np
import pandas as pd


def plain(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(type(value).__name__)


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=plain, allow_nan=False).encode()).hexdigest()


def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def read_jsonl(path):
    return [json.loads(s) for s in Path(path).read_text(encoding='utf-8').splitlines() if s.strip()]


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=plain, allow_nan=False) + '\n', encoding='utf-8')
    temp.replace(path)


def freeze(path, identity):
    identity = json.loads(json.dumps(identity, default=plain, sort_keys=True, allow_nan=False))
    if Path(path).is_file():
        assert read_json(path) == identity, f'Configuration changed; preserve the old run and use a new RUN_ID: {path}'
    else:
        save_json(path, identity)
    return digest(identity)


def cached_json(directory, key, identity, compute):
    path = Path(directory) / (digest(str(key))[:24] + '.json')
    expected = digest(identity)
    if path.is_file():
        old = read_json(path)
        assert old['key'] == str(key) and old['identity'] == expected, f'Incompatible cache: {path}'
        return old['result']
    value = compute()
    save_json(path, dict(key=str(key), identity=expected, result=value))
    return value


def bootstrap(values, key='', replicates=10000):
    x = np.asarray(values, dtype=float)
    assert x.ndim == 1 and len(x) and np.isfinite(x).all()
    rng = np.random.default_rng(int(digest(str(key))[:16], 16))
    draws = np.concatenate([rng.choice(x, (min(250, replicates-i), len(x)), replace=True).mean(1)
                            for i in range(0, replicates, 250)])
    return dict(n=len(x), mean=float(x.mean()), lower=float(np.quantile(draws, .025)),
                upper=float(np.quantile(draws, .975)),
                p_value=float((1 + (draws <= 0).sum()) / (replicates + 1)))


def bh(values):
    p = np.asarray(values, dtype=float)
    assert np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()
    order = np.argsort(p)
    q = np.empty_like(p)
    previous = 1.
    for rank in range(len(p)-1, -1, -1):
        i = order[rank]
        previous = min(previous, p[i]*len(p)/(rank+1))
        q[i] = previous
    return q


def paired(left, right, id_col, value):
    assert left[id_col].is_unique and right[id_col].is_unique
    assert set(left[id_col]) == set(right[id_col]), 'Different paired support'
    out = left[[id_col, value]].merge(right[[id_col, value]], on=id_col, validate='one_to_one', suffixes=('_l', '_r'))
    return out[value+'_l'].to_numpy(float) - out[value+'_r'].to_numpy(float)


def semantic_scores(record):
    if 'semantic_logprobs' in record:
        return {str(k): float(v) for k, v in record['semantic_logprobs'].items()}
    values = record['candidate_logprobs']
    if isinstance(values, dict):
        return {str(record['label_to_semantic'][k]): float(v) for k, v in values.items()}
    return dict(zip(map(str, record['candidate_semantics']), map(float, values)))


def correct_margin(record):
    values = semantic_scores(record)
    gold = str(record['gold_semantic'])
    assert gold in values and len(values) >= 2
    return values[gold] - max(v for k, v in values.items() if k != gold)


def random_unit(shape, key):
    r = np.random.default_rng(int(digest(key)[:16], 16)).normal(size=shape)
    return r / np.linalg.norm(r)


def coordinate_clamp(pre, untreated, axis, cast):
    h, base, u = [np.asarray(x, dtype=np.float64) for x in (pre, untreated, axis)]
    u = u / np.linalg.norm(u)
    a = float(np.sum((h-base)*u))
    return np.asarray(cast(h-a*u), dtype=np.float64), a


def native_norm_edit(pre, direction, norm, cast, tolerance=.02, steps=48):
    h = np.asarray(pre, dtype=np.float64)
    r = np.asarray(direction, dtype=np.float64)
    assert np.isfinite(h).all() and np.isfinite(r).all() and norm >= 0 and np.linalg.norm(r) > 0
    r /= np.linalg.norm(r)
    if norm == 0:
        return h.copy(), dict(actual_norm=0., target_norm=0., relative_error=0., matched=True)
    lo, hi = 0., max(float(norm), 1e-12)
    for _ in range(32):
        if np.linalg.norm(np.asarray(cast(h+hi*r))-h) >= norm:
            break
        hi *= 2
    best, error = h.copy(), 1.
    for _ in range(steps):
        mid = (lo+hi)/2
        candidate = np.asarray(cast(h+mid*r), dtype=np.float64)
        actual = float(np.linalg.norm(candidate-h))
        current = abs(actual-norm)/norm
        if current < error:
            best, error = candidate.copy(), current
        if error <= tolerance:
            break
        if actual < norm:
            lo = mid
        else:
            hi = mid
    return best, dict(actual_norm=float(np.linalg.norm(best-h)), target_norm=float(norm),
                      relative_error=float(error), matched=bool(error <= tolerance))


def equal_removal(pre, untreated, axis, random_direction, cast, tolerance=.02):
    target, coefficient = coordinate_clamp(pre, untreated, axis, cast)
    target_norm = float(np.linalg.norm(target-np.asarray(pre)))
    sign = 1. if coefficient >= 0 else -1.
    post, audit = native_norm_edit(pre, -sign*np.asarray(random_direction), target_norm, cast, tolerance)
    audit.update(reference_coefficient=coefficient, reference_actual_norm=target_norm)
    return post, audit


def geometry_signature(stacks):
    a = tuple(sorted((str(s['side']), int(s['distance']), int(s['units'])) for s in stacks))
    b = tuple(sorted(('right' if side == 'left' else 'left', d, u) for side, d, u in a))
    return min(a, b)  # mirrors belong to the same physical family


def geometries_in(value):
    found = set()
    if isinstance(value, dict):
        stacks = value.get('stacks')
        if isinstance(stacks, list) and stacks and all(isinstance(s, dict) and {'side','distance','units'} <= set(s) for s in stacks):
            found.add(geometry_signature(stacks))
        # Some prior matched cohorts store geometry instead of stack records.
        geom = value.get('pair_geometry')
        if isinstance(geom, (list, tuple)) and len(geom) == 4 and all(isinstance(x, (int,float)) for x in geom):
            u1,d1,u2,d2 = map(int, geom)
            found.add(geometry_signature([dict(side='left',distance=d1,units=u1),dict(side='right',distance=d2,units=u2)]))
        if isinstance(geom, (list, tuple)) and len(geom)==2 and all(isinstance(x,(list,tuple)) and len(x)==2 for x in geom):
            (u1,d1),(u2,d2)=geom
            found.add(geometry_signature([dict(side='left',distance=int(d1),units=int(u1)),dict(side='right',distance=int(d2),units=int(u2))]))
        for child in value.values():
            found.update(geometries_in(child))
    elif isinstance(value, list):
        for child in value:
            found.update(geometries_in(child))
    return found


def physics(stacks):
    units = {s:sum(int(x['units']) for x in stacks if x['side']==s) for s in ('left','right')}
    dist = {s:sum(int(x['distance']) for x in stacks if x['side']==s) for s in ('left','right')}
    torque = {s:sum(int(x['units'])*int(x['distance']) for x in stacks if x['side']==s) for s in ('left','right')}
    out = lambda x:'left' if x>0 else 'right' if x<0 else 'balanced'
    net=torque['left']-torque['right']; w=units['left']-units['right']; d=dist['left']-dist['right']
    return dict(net_torque=net,outcome=out(net),weight_score=w,distance_score=d,
                normalized_margin=abs(net)/max(1,sum(torque.values())),
                left_units=units['left'],right_units=units['right'],left_torque=torque['left'],right_torque=torque['right'],
                strategy_predictions=dict(exact_torque=out(net),weight_only=out(w),distance_only=out(d)))


def exclusion_snapshot(project, followup_root, dataset_dir=None):
    lock=Path(followup_root)/'prior_geometry_snapshot.json'
    if lock.is_file():
        old=read_json(lock)
        for path, sha in old['files'].items():
            assert file_hash(Path(path))==sha, f'Prior input changed: {path}'
        return {tuple(tuple(x) for x in sig) for sig in old['signatures']}, old
    paths={Path(dataset_dir or project/'data/dataset_v1')/'scenes.jsonl'}
    for pattern in ('*manifest*.jsonl','*support*.jsonl','*geometries*.jsonl'):
        paths.update(p for p in (project/'results').rglob(pattern) if Path(followup_root) not in p.parents)
    excluded=set(); hashes={}
    for path in sorted(paths):
        assert path.is_file(), f'Missing required geometry input: {path}'
        excluded.update(geometries_in(read_jsonl(path)))
        hashes[str(path.resolve())]=file_hash(path)
    assert excluded, 'No prior physical geometry was indexed'
    old=dict(files=hashes,signatures=sorted(excluded),mirror_canonical=True)
    save_json(lock,old)
    return excluded,old


def inverse_families(n, seed, excluded):
    assert n%4==0
    pool=defaultdict(list)
    for dl in combinations(range(1,7),2):
        for dr in combinations(range(1,7),2):
            for units in product(range(1,6), repeat=4):
                stacks=[dict(side=side,distance=d,units=u) for side,d,u in zip(('left','left','right','right'),dl+dr,units)]
                f=physics(stacks)
                pool[dl+dr].append((stacks,f))
    candidates=[]
    for distances,rows in pool.items():
        by_net=defaultdict(list)
        for stacks,f in rows:
            if f['outcome']==f['strategy_predictions']['weight_only'] and f['outcome']!='balanced':
                by_net[f['net_torque']].append((stacks,f))
        for stacks,f in rows:
            margin=f['normalized_margin']; difficulty='hard' if .08<=margin<.22 else 'moderate' if .22<=margin<.45 else None
            if (difficulty and f['outcome']!='balanced' and f['strategy_predictions']['weight_only'] not in ('balanced',f['outcome'])
                    and f['strategy_predictions']['distance_only']==f['outcome'] and abs(f['distance_score'])>=2):
                sources=by_net.get(f['net_torque'],[])
                if sources:candidates.append((digest([seed,stacks]),difficulty,stacks,f,sources))
    candidates.sort(key=lambda r:r[0]); used=set(excluded); counts=Counter(); result=[]
    for _,difficulty,stacks,f,sources in candidates:
        stratum=(difficulty,f['outcome'])
        if counts[stratum]>=n//4 or geometry_signature(stacks) in used:continue
        valid=[(s,g) for s,g in sources if geometry_signature(s) not in used|{geometry_signature(stacks)}]
        if not valid:continue
        valid.sort(key=lambda x:(sum(abs(a['units']-b['units']) for a,b in zip(stacks,x[0])),digest([seed,x[0]])))
        src,sf=valid[0];fid='fixed_'+digest([seed,stacks,src])[:20]
        roles={name:dict(scene_id=fid+'__'+name,image_filename=fid+'__'+name+'.png',stacks=s,features=g)
               for name,s,g in [('base',stacks,f),('aligned_source',src,sf)]}
        result.append(dict(family_id=fid,remapping_split='new_fixed_confirmation',roles=roles,margin_bin=difficulty))
        used.update([geometry_signature(stacks),geometry_signature(src)]);counts[stratum]+=1
        if len(result)==n:break
    assert len(result)==n and set(counts.values())=={n//4}, (len(result),counts)
    return sorted(result,key=lambda r:r['family_id'])


def repair_families(n, seed, excluded):
    assert n%4==0
    candidates=defaultdict(list)
    for ul,dl,ur,dr in product(range(1,9),range(1,7),range(1,9),range(1,7)):
        s=[dict(side='left',units=ul,distance=dl),dict(side='right',units=ur,distance=dr)]
        f=physics(s)
        if f['net_torque']==0:continue
        if ul==ur and dl!=dr:kind='equal_weight_distance'
        elif f['strategy_predictions']['weight_only'] not in ('balanced',f['outcome']):kind='weight_distance_conflict'
        elif f['strategy_predictions']['weight_only']==f['outcome'] and f['strategy_predictions']['distance_only']==f['outcome']:kind='aligned_cues'
        else:continue
        candidates[kind].append(s)
    for units in product(range(1,6),repeat=4):
        for distances in [((1,3),(2,5)),((2,5),(1,4)),((1,6),(3,5)),((2,4),(3,6))]:
            s=[dict(side=side,distance=d,units=u) for side,d,u in zip(('left','left','right','right'),distances[0]+distances[1],units)]
            if physics(s)['net_torque']!=0:candidates['multi_stack_sum'].append(s)
    used=set(excluded); result=[]
    for kind in ('equal_weight_distance','weight_distance_conflict','multi_stack_sum','aligned_cues'):
        rows=sorted(candidates[kind],key=lambda s:digest([seed,kind,s])); count=0
        for stacks in rows:
            sig=geometry_signature(stacks)
            if sig in used:continue
            fid='repair_'+digest([seed,kind,sig])[:20];scenes=[]
            for mirror in (False,True):
                current=[dict(s,side=('right' if s['side']=='left' else 'left')) if mirror else dict(s) for s in stacks]
                for style in ('standard','alternate'):
                    sid=f'{fid}__m{int(mirror)}__{style}'
                    scenes.append(dict(scene_id=sid,image_filename=sid+'.png',stacks=current,features=physics(current),style=style,mirror=mirror))
            result.append(dict(family_id=fid,stratum=kind,scenes=scenes));used.add(sig);count+=1
            if count==n//4:break
        assert count==n//4,(kind,count,'Insufficient unused geometries; do not silently change task ranges')
    return sorted(result,key=lambda r:r['family_id'])


"""Pure numerical kernels embedded in the E1 notebook; cast is supplied by the runtime."""
import hashlib
import numpy as np


def readout_value(hidden, mean, scale, coef, intercept):
    return float(((np.asarray(hidden, dtype=np.float64) - mean) / scale) @ coef + intercept)


def numerical_correction(hidden, target, mean, scale, coef, intercept, cast,
                         tolerance=0.05, steps=32):
    """Calibrate only the external readout; downstream always receives native dtype.

    The initial edit minimizes standardized-space norm. Subsequent quantization
    calibration need not be the globally smallest native-dtype perturbation.
    """
    h = np.asarray(hidden, dtype=np.float64)
    a = np.asarray(coef, dtype=np.float64)
    scale = np.asarray(scale, dtype=np.float64)
    assert np.all(scale > 0) and float(a @ a) > 0 and steps > 0
    initial = readout_value(h, mean, scale, a, intercept)
    unit = scale * a / float(a @ a)
    planned = (target - initial) * unit
    # Preserve an exact zero edit for already-in-tolerance examples.
    best = h.copy()
    error = abs(initial - target)
    if error <= tolerance:
        return best, dict(planned_delta_norm=0.0, residual=error, achieved=True,
                          iterations=0, originally_within_tolerance=True)
    work = planned.copy()
    for iteration in range(steps):
        candidate = np.asarray(cast(h + work), dtype=np.float64)
        value = readout_value(candidate, mean, scale, a, intercept)
        candidate_error = abs(target - value)
        if candidate_error < error:
            best, error = candidate.copy(), candidate_error
        if error <= tolerance:
            break
        work += (target - value) * unit
    return best, dict(planned_delta_norm=float(np.linalg.norm(planned)),
                      residual=float(error), achieved=bool(error <= tolerance),
                      iterations=iteration + 1, originally_within_tolerance=False)


def deterministic_direction(shape, key, scale, coef, orthogonal=False):
    seed = int(hashlib.sha256(str(key).encode()).hexdigest()[:16], 16)
    z = np.random.default_rng(seed).normal(size=shape)
    if orthogonal:
        z -= coef * float(z @ coef) / float(coef @ coef)
    raw = z * scale
    assert np.linalg.norm(raw) > 0
    return raw / np.linalg.norm(raw)


def match_native_norm(hidden, direction, target_norm, cast, tolerance=0.02, steps=48):
    """Bounded amplitude search, matching actual raw residual norm after casting.

    A discrete native dtype can make exact norm matching impossible. Return the
    best candidate and an explicit failure flag; never silently call it matched.
    """
    h = np.asarray(hidden, dtype=np.float64)
    direction = np.asarray(direction, dtype=np.float64)
    assert target_norm >= 0 and np.linalg.norm(direction) > 0
    direction = direction / np.linalg.norm(direction)
    if target_norm == 0:
        return h.copy(), dict(norm_error_relative=0.0, norm_matched=True, amplitude=0.0)
    lo, hi = 0.0, max(target_norm, 1e-12)
    for _ in range(32):
        if np.linalg.norm(np.asarray(cast(h + hi * direction)) - h) >= target_norm:
            break
        hi *= 2
    best, best_error, best_amplitude = h.copy(), 1.0, 0.0
    for _ in range(steps):
        amplitude = (lo + hi) / 2
        candidate = np.asarray(cast(h + amplitude * direction), dtype=np.float64)
        norm = float(np.linalg.norm(candidate - h))
        error = abs(norm - target_norm) / target_norm
        if error < best_error:
            best, best_error, best_amplitude = candidate.copy(), error, amplitude
        if error <= tolerance:
            break
        if norm < target_norm:
            lo = amplitude
        else:
            hi = amplitude
    return best, dict(norm_error_relative=float(best_error),
                      norm_matched=bool(best_error <= tolerance), amplitude=float(best_amplitude))


def scene_bootstrap(values, seed=20260913, replicates=10000):
    """Rows must already be independent scene/family means, not mappings/seeds."""
    values = np.asarray(values, dtype=float)
    assert values.ndim == 1 and len(values) and np.isfinite(values).all()
    rng = np.random.default_rng(seed)
    means = []
    for start in range(0, replicates, 250):
        size = min(250, replicates - start)
        means.extend(values[rng.integers(0, len(values), (size, len(values)))].mean(axis=1))
    return dict(mean=float(values.mean()), n=len(values),
                lower=float(np.quantile(means, .025)), upper=float(np.quantile(means, .975)))


def paired_on_ids(left, right, id_column, value_column):
    """Do not silently turn partial overlap or duplicate rows into a paired comparison."""
    assert left[id_column].is_unique and right[id_column].is_unique
    assert set(left[id_column]) == set(right[id_column]), 'Different paired support'
    joined = left[[id_column, value_column]].merge(
        right[[id_column, value_column]], on=id_column, suffixes=('_left', '_right'), validate='one_to_one')
    return joined[value_column + '_left'].to_numpy() - joined[value_column + '_right'].to_numpy()


"""Small registered-output packet; never include stale files from another run."""


class DiagnosticPacket:
    def __init__(self, root, identity):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        freeze(self.root/'protocol.json', identity)
        self.outputs = {self.root/'protocol.json'}
        self.inputs = {}

    def track(self, path):
        path = Path(path)
        self.inputs[str(path)] = file_hash(path)
        return path

    def csv(self, name, frame):
        path = self.root/name
        frame.to_csv(path, index=False, encoding='utf-8-sig')
        self.outputs.add(path)
        return path

    def json(self, name, value):
        path = self.root/name
        save_json(path, value)
        self.outputs.add(path)
        return path

    def text(self, name, value):
        path = self.root/name
        path.write_text(value, encoding='utf-8')
        self.outputs.add(path)
        return path

    def finish(self, summary, report, archive_name, auto_download=True):
        import zipfile
        changed = [p for p, h in self.inputs.items() if not Path(p).is_file() or file_hash(p) != h]
        summary = dict(summary, source_files_unchanged=not changed, changed_inputs=changed)
        if changed:
            summary['status'] = 'inputs_changed_results_not_comparable'
        self.json('input_manifest.json', self.inputs)
        self.json('RUN_COMPLETE.json', summary)
        self.text('REPORT_ZH.md', report)
        archive = self.root/archive_name
        temporary = archive.with_suffix('.zip.tmp')
        with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
            for path in sorted(self.outputs):
                bundle.write(path, str(path.relative_to(self.root)))
        temporary.replace(archive)
        print(json.dumps(summary, ensure_ascii=False, indent=2))
        print('完整结果包：', archive)
        if auto_download:
            try:
                from google.colab import files
                files.download(str(archive))
            except Exception as error:
                print('自动下载未触发，请从以上 Drive 路径手动下载：', type(error).__name__, str(error))
        return archive


def diagnostic_show(title, frame, limit=30):
    print('\n'+title)
    if frame.empty:
        print('无记录或此项输入缺失；请查看 coverage。')
        return
    with pd.option_context('display.max_rows', limit+2, 'display.max_columns', None, 'display.max_colwidth', 90):
        display(frame.head(limit))
    if len(frame) > limit:
        print(f'这里显示前 {limit} 行；结果 ZIP 保存全部 {len(frame)} 行。')


"""CPU helpers for matched-capture readout verification; no model execution."""
from collections.abc import Mapping
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import sklearn
import scipy


def fixed_variant(scene_id, seed=20260820):
    key=hashlib.sha256(f'{seed}:variant:{scene_id}'.encode()).hexdigest()
    return 'permutation_'+str(int(key[:8],16)%6)


def probe_parameters(probe):
    mean,scale,coef=[np.asarray(probe[k],np.float32) for k in ('mean','scale','coef')]
    assert mean.ndim==1 and mean.shape==scale.shape==coef.shape
    assert (scale>0).all() and all(np.isfinite(a).all() for a in (mean,scale,coef))
    assert float(np.asarray(coef,np.float64)@coef)>0 and np.isfinite(probe['intercept'])
    return mean,scale,coef,float(probe['intercept'])


def decode_with(hidden,probe):
    return readout_value(hidden,*probe_parameters(probe))


def readout_class(value,threshold=.5):
    return 'left' if value>threshold else 'right' if value < -threshold else 'balanced'


def six_mapping_mean(native):
    # Preserve the original per-mapping FP16 capture and FP16 scene-mean rule.
    a=np.asarray(native,np.float32)
    assert a.ndim==2 and a.shape[0]==6 and np.isfinite(a).all()
    return a.astype(np.float16).astype(np.float32).mean(axis=0).astype(np.float16).astype(np.float32)


def fit_fixed_probe(states,targets,scene_ids,layer,alpha,provenance):
    x=np.asarray(states,np.float32);y=np.asarray(targets,float)
    assert len(x)==len(y)==len(scene_ids)==len(set(scene_ids)) and np.isfinite(x).all() and np.isfinite(y).all()
    scaler=StandardScaler()
    transformed=scaler.fit_transform(x)
    estimator=Ridge(alpha=float(alpha),solver='lsqr',max_iter=5000).fit(transformed,y)
    result=dict(layer=int(layer),alpha=float(alpha),mean=scaler.mean_.astype(np.float32),
        scale=scaler.scale_.astype(np.float32),coef=estimator.coef_.astype(np.float32),intercept=float(estimator.intercept_),
        fit_scene_ids=list(scene_ids),fit_provenance=provenance)
    probe_parameters(result)
    return json.loads(json.dumps(result,default=plain,allow_nan=False))


def aligned_cache(root,stage,key,identity,compute):
    folder=Path(root)/'cache'/stage/digest(key)[:24]
    folder.mkdir(parents=True,exist_ok=True)
    meta_path=folder/'record.json';array_path=folder/'states.npz'
    if meta_path.is_file():
        saved=read_json(meta_path)
        assert saved['identity']==identity and saved['key']==key, 'Cache identity mismatch; use a new RUN_ID.'
        assert array_path.is_file() and saved['states_sha256']==file_hash(array_path)
        with np.load(array_path,allow_pickle=False) as z:
            arrays={k:z[k].copy() for k in z.files}
        return saved['records'],arrays,True
    records,arrays=compute()
    assert all(np.isfinite(a).all() for a in arrays.values())
    temp=array_path.with_suffix('.npz.tmp')
    with temp.open('wb') as f:np.savez_compressed(f,**arrays)
    temp.replace(array_path)
    save_json(meta_path,dict(identity=identity,key=key,records=records,states_sha256=file_hash(array_path)))
    return read_json(meta_path)['records'],arrays,False


def save_arrays(packet,name,**arrays):
    path=packet.root/name;temp=path.with_suffix('.npz.tmp')
    with temp.open('wb') as f:np.savez_compressed(f,**arrays)
    temp.replace(path);packet.outputs.add(path)
    return file_hash(path)


def require_same_pre(before,reference,input_hash,reference_hash):
    assert input_hash==reference_hash, 'Prepared inputs changed within a paired comparison.'
    assert np.array_equal(before,reference), 'Pre-intervention hidden state changed within a paired comparison.'


def candidate_margin(record):
    scores=record['semantic_logprobs'];gold=record['gold_semantic']
    return float(scores[gold]-max(v for k,v in scores.items() if k!=gold))


def random_validity(actual_norm,target_norm,tolerance=.02):
    assert np.isfinite(actual_norm) and np.isfinite(target_norm) and actual_norm>=0 and target_norm>=0
    # A nonzero edit cannot match a zero reference; its relative error is undefined.
    if target_norm==0:return bool(actual_norm==0),0. if actual_norm==0 else None
    relative=abs(actual_norm-target_norm)/target_norm
    return bool(relative<=tolerance),float(relative)


def summarize_test(records,expected=1007):
    frame=pd.DataFrame(records)
    assert len(frame)==expected and frame.scene_id.is_unique
    rows=[]
    for group,sub in [('all',frame),*[(str(k),v) for k,v in frame.groupby('problem_type')]]:
        for probe in ('aligned','historical'):
            correct=sub[probe+'_readout_correct'].astype(bool)
            rows.append(dict(group=group,probe=probe,scenes=len(sub),readout_correct=int(correct.sum()),
                readout_accuracy=float(correct.mean()),model_correct=int(sub.correct.sum()),model_accuracy=float(sub.correct.mean()),
                readout_correct_model_wrong=int((correct&~sub.correct.astype(bool)).sum()),
                readout_mae=float((sub[probe+'_readout']-sub.net_torque).abs().mean()),
                interpretation='fixed_single_mapping_on_reused_test_scenes'))
    return pd.DataFrame(rows)


def summarize_interventions(records,selected_ids,control_ids,replicates=10000):
    frame=pd.DataFrame(records)
    assert not frame.duplicated(['scene_id','condition','random_seed']).any()
    assert set(frame.condition)=={'zero','oracle','random'}
    selected_ids=set(selected_ids);control_ids=set(control_ids)
    assert control_ids<=selected_ids
    zero=frame[frame.condition.eq('zero')].set_index('scene_id').sort_index()
    oracle=frame[frame.condition.eq('oracle')].set_index('scene_id').sort_index()
    random=frame[frame.condition.eq('random')]
    assert set(zero.index)==set(oracle.index)==selected_ids and zero.index.is_unique and oracle.index.is_unique
    assert set(random.scene_id)==control_ids
    assert all(set(g.random_seed)=={0,1,2,3} and len(g)==4 for _,g in random.groupby('scene_id'))
    ci=lambda values:scene_bootstrap(np.asarray(values,float),replicates=replicates)
    summaries=[]
    for name,ids in [('all_304',sorted(selected_ids)),('tilt_263',list(zero.index[zero.net_torque.ne(0)])),
                     ('balance_41',list(zero.index[zero.net_torque.eq(0)]))]:
        if not ids:continue
        a=zero.loc[ids];b=oracle.loc[ids]
        repaired=(~a.correct.astype(bool)&b.correct.astype(bool));harmed=(a.correct.astype(bool)&~b.correct.astype(bool))
        change=b.correct.astype(float)-a.correct.astype(float)
        interval=ci(change)
        summaries.append(dict(group=name,scenes=len(ids),before_correct=int(a.correct.sum()),after_correct=int(b.correct.sum()),
            original_errors=int((~a.correct.astype(bool)).sum()),repaired=int(repaired.sum()),harmed=int(harmed.sum()),
            answer_changed=int((a.predicted_semantic!=b.predicted_semantic).sum()),
            readout_correct_before=int(a.readout_correct.sum()),readout_correct_after=int(b.readout_correct.sum()),
            readout_mae_before=float((a.decoded_post-a.net_torque).abs().mean()),
            readout_mae_after=float((b.decoded_post-b.net_torque).abs().mean()),
            oracle_achieved=int(b.achieved.eq(True).sum()),oracle_failed=int(b.achieved.ne(True).sum()),
            accuracy_change=interval['mean'],lower=interval['lower'],upper=interval['upper'],
            excludes_5pp_gain_for_all_attempts=bool(interval['upper']<.05)))
    contrasts=[]
    def add_contrast(scope,ids,kind):
        ids=sorted(ids)
        if not ids:return
        a=zero.loc[ids];o=oracle.loc[ids]
        if kind=='oracle_minus_zero':
            acc=o.correct.astype(float)-a.correct.astype(float)
            margin=o.correct_answer_margin-a.correct_answer_margin
        else:
            r=random[random.scene_id.isin(ids)].groupby('scene_id')[['correct','correct_answer_margin']].mean().loc[ids]
            acc=o.correct.astype(float)-r.correct
            margin=o.correct_answer_margin-r.correct_answer_margin
        for metric,x in [('accuracy',acc),('correct_answer_margin',margin)]:
            contrasts.append(dict(scope=scope,comparison=kind,metric=metric,**ci(x),
                interpretation='paired_protocol_robustness_not_new_independent_confirmation'))
    add_contrast('all_304_attempted',selected_ids,'oracle_minus_zero')
    succeeded=set(oracle.index[oracle.achieved.eq(True)])
    add_contrast('oracle_achieved_subset',succeeded,'oracle_minus_zero')
    add_contrast('all_72_attempted',control_ids,'oracle_minus_random')
    valid=set(random.groupby('scene_id').valid_control.all().loc[lambda x:x].index)&succeeded
    add_contrast('oracle_achieved_all_four_random_valid',valid,'oracle_minus_random')
    technical=dict(oracle_attempted=len(oracle),oracle_achieved=len(succeeded),oracle_failed=len(oracle)-len(succeeded),
        zero_or_tiny_edits=int((oracle.actual_delta_norm<=1e-10).sum()),random_attempted=len(random),
        random_invalid=int((~random.valid_control.astype(bool)).sum()),common_valid_control_scenes=len(valid),
        all_attempts_retained=True)
    return pd.DataFrame(summaries),pd.DataFrame(contrasts),technical


In [ ]:
%%cell_progress 5/17 Prepare existing Dataset v1
cell_stage('Checking local and Drive dataset caches')
from zipfile import ZipFile

REQUIRED_DATASET_FILES = {
    'dataset_config.json', 'scenes.jsonl', 'counterfactual_pairs.jsonl',
    'questions.jsonl', 'splits.json', 'statistics.json',
}
DATASET_DIR = Path('/content/visualtorque_dataset_v1')
DATASET_ARCHIVE = PROJECT_ROOT / 'data' / 'dataset_v1.zip'
LEGACY_DATASET_DIR = PROJECT_ROOT / 'data' / 'dataset_v1'

def dataset_complete(path):
    return path.is_dir() and all(
        (path / name).is_file() for name in REQUIRED_DATASET_FILES
    )

if not dataset_complete(DATASET_DIR):
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    if DATASET_ARCHIVE.is_file():
        cell_stage('Extracting Dataset v1 archive')
        with ZipFile(DATASET_ARCHIVE, 'r') as bundle:
            members = [m for m in bundle.infolist() if not m.is_dir()]
            root = DATASET_DIR.resolve()
            for member in tqdm(members, desc='Extracting Dataset v1', unit='file'):
                target = (DATASET_DIR / member.filename).resolve()
                if root not in target.parents:
                    raise ValueError(f'Unsafe archive path: {member.filename}')
                bundle.extract(member, DATASET_DIR)
    elif dataset_complete(LEGACY_DATASET_DIR):
        DATASET_DIR = LEGACY_DATASET_DIR
    else:
        raise FileNotFoundError(
            'Run the Dataset v1 build/cache cell in the behavior notebook first.'
        )
assert dataset_complete(DATASET_DIR)
print('Dataset:', DATASET_DIR)


In [ ]:
%%cell_progress 6/17 Select model and preserve all previous experiments
MODEL_KEY='gemma12b'  # @param ["gemma12b", "qwen9b"]
RUN_ID='main'
AUTO_DOWNLOAD=True
SPECS={
 'gemma12b':dict(model_id='google/gemma-4-12B-it',slug='google__gemma-4-12B-it',layers=48,layer=7,alpha=100.,
     revision='707f0a3b8a3c7ad586ed01e27eafbad8a27dd0f7'),
 'qwen9b':dict(model_id='Qwen/Qwen3.5-9B',slug='Qwen__Qwen3.5-9B',layers=32,layer=13,alpha=.01,
     revision='c202236235762e1c871ad0ccb60c8ee5ba337b9a')}
assert MODEL_KEY in SPECS, 'This notebook is only for Gemma-12B and Qwen-9B.'
SPEC=SPECS[MODEL_KEY];MODEL_ID=SPEC['model_id'];MODEL_SLUG=SPEC['slug'];MODEL_REVISION=SPEC['revision']
EXPECTED_LAYER_COUNT=SPEC['layers'];SELECTED_LAYER=SPEC['layer'];FROZEN_ALPHA=SPEC['alpha']
CAPTURE_LAYERS=(SELECTED_LAYER,);MODEL_DTYPE='bfloat16';DEVICE_MAP='auto'
READOUT_TOLERANCE=.05;NORM_TOLERANCE=.02;BALANCE_THRESHOLD=.5;RANDOM_SEEDS=(0,1,2,3)
MODEL_ROOT=PROJECT_ROOT/'results/cross_model_aligned_readout_v1'/RUN_ID/MODEL_SLUG
PROTOCOL=dict(schema='aligned-readout-validation-1.0',implementation=IMPLEMENTATION_SHA256,model=MODEL_KEY,
    model_id=MODEL_ID,revision=MODEL_REVISION,layer=SELECTED_LAYER,alpha=FROZEN_ALPHA,batch_size=1,dtype=MODEL_DTYPE,
    fit_scenes=254,fit_mapping_count=6,test_scenes=1007,test_mapping='historically_fixed_single_mapping',
    intervention_scenes=304,random_control_scenes=72,random_seeds=list(RANDOM_SEEDS),
    storage='native states preserved; fit uses per-mapping FP16 then FP32 six-map mean then FP16',
    fit='StandardScaler and ridge lsqr max_iter=5000; refit original 254 with fixed layer and alpha',
    readout_tolerance=READOUT_TOLERANCE,norm_tolerance=NORM_TOLERANCE,balance_threshold=BALANCE_THRESHOLD,
    accuracy_sesoi=.05,bootstrap_unit='scene',bootstrap_replicates=10000,oracle_uses_true_torque=True,
    primary_comparison='oracle minus fresh zero accuracy, all 304 attempts',
    control_comparison='oracle minus mean of four raw-isotropic equal-actual-norm directions, original 72 scenes',
    new_independent_confirmation=False,historical_activation_reuse=False,historical_test_cache_reuse=False)
packet=DiagnosticPacket(MODEL_ROOT/'setup',PROTOCOL)
ready=True;issues=[];stages=[];NEW_FORWARD_COUNT=0;CACHE_HITS=0
training_records=[];training_arrays=[];training_meta=[];test_records=[];test_arrays=[]
intervention_records=[];intervention_arrays=[];intervention_keys=[];dev_checks=[]
eval_summary=pd.DataFrame();effect_summary=pd.DataFrame();contrasts=pd.DataFrame();technical={}
def record_issue(stage,error):
    global ready
    ready=False
    item=dict(stage=stage,error_type=type(error).__name__,reason=str(error))
    issues.append(item)
    print('此阶段未完成；请继续运行到最后打包。',item)
print('模型：',MODEL_ID,'；固定层：',SELECTED_LAYER,'；训练、测试和干预全部逐张处理。')
print('原实验目录不改动；新结果保存于：',MODEL_ROOT)


In [ ]:
%%cell_progress 7/17 Define the shared capture and intervention functions
def output_hidden(output):
    if torch.is_tensor(output):
        hidden = output
    elif isinstance(output, (tuple, list)) and torch.is_tensor(output[0]):
        hidden = output[0]
    else:
        raise TypeError(f'Unsupported decoder output: {type(output).__name__}')
    assert hidden.ndim == 3
    return hidden

def replace_hidden(output, value):
    if torch.is_tensor(output):
        return value
    if isinstance(output, tuple):
        return (value,) + output[1:]
    if isinstance(output, list):
        return [value] + output[1:]
    raise TypeError(type(output).__name__)

def tensor_signature(value):
    if torch.is_tensor(value):
        v = value.detach().contiguous().cpu()
        raw = v.reshape(-1).view(torch.uint8).numpy().tobytes()
        return dict(dtype=str(v.dtype), shape=list(v.shape), sha256=hashlib.sha256(raw).hexdigest())
    if isinstance(value, (tuple, list)):
        return [tensor_signature(v) for v in value]
    # HF BatchFeature/BatchEncoding are mapping wrappers, not dict subclasses.
    # Inspect every contained value without changing the prepared model inputs.
    if isinstance(value, Mapping):
        return {str(k): tensor_signature(v) for k, v in sorted(value.items())}
    if value is None or isinstance(value, (str, bool, int, float)):
        return value
    raise TypeError('Untracked prepared-input value: '+type(value).__name__)

"""One forward convention for training, test, zero replay and all interventions."""


def aligned_forward(example,edit=None,reference=None,reference_input_hash=None):
    global NEW_FORWARD_COUNT
    plan=adapter._candidate_plan(example)
    assert plan is not None
    prefix,tokens=plan
    inputs=adapter._append_text_prefix(adapter._to_device(adapter._template(
        [example.messages()],add_generation_prompt=True)),prefix)
    assert inputs['input_ids'].shape[0]==1 and bool(inputs['attention_mask'][0,-1])
    fingerprint=digest(tensor_signature(inputs))
    if reference_input_hash is not None:assert fingerprint==reference_input_hash
    captured={};audit={};handle=None
    def hook(_module,_inputs,output):
        hidden=output_hidden(output)
        assert hidden.dtype==torch.bfloat16 and hidden.shape[0]==1
        pre=hidden[0,-1].detach().float().cpu().numpy().copy()
        if reference is not None:require_same_pre(pre,reference,fingerprint,reference_input_hash)
        captured['pre']=pre
        if edit is None:
            captured['post']=pre.copy()
            return None
        cast=lambda x:torch.as_tensor(np.asarray(x).copy(),dtype=hidden.dtype).float().cpu().numpy()
        proposed,details=edit(pre,cast)
        modified=hidden.clone()
        modified[0,-1]=torch.as_tensor(proposed,dtype=hidden.dtype,device=hidden.device)
        post=modified[0,-1].detach().float().cpu().numpy().copy()
        captured['post']=post
        audit.update(details)
        audit.update(actual_delta_norm=float(np.linalg.norm(post-pre)),
            standardized_delta_norm=float(np.linalg.norm((post-pre)/PROBE_SCALE)),
            decoded_pre=decode(pre),decoded_post=decode(post),hidden_norm=float(np.linalg.norm(pre)))
        return replace_hidden(output,modified)
    handle=DECODER_LAYERS[SELECTED_LAYER].register_forward_hook(hook)
    try:
        with torch.inference_mode():
            NEW_FORWARD_COUNT+=1
            result=adapter.model(**inputs,use_cache=False)
            scores=torch.log_softmax(result.logits[0,-1].float(),dim=-1)
            values={str(label):float(scores[tokens[label]].cpu()) for label in example.labels}
    finally:
        handle.remove()
    assert set(captured)=={'pre','post'} and all(np.isfinite(v).all() for v in captured.values())
    label=max(example.labels,key=lambda k:values[str(k)])
    predicted=str(example.label_to_semantic[label])
    record=dict(example_id=str(example.example_id),scene_id=str(example.scene_id),variant=str(example.variant),
        split=str(example.split),problem_type=str(example.problem_type),net_torque=int(example.net_torque),
        gold_semantic=str(example.gold_semantic),predicted_semantic=predicted,predicted_label=str(label),
        correct=predicted==str(example.gold_semantic),input_fingerprint=fingerprint,
        label_to_semantic={str(k):str(v) for k,v in example.label_to_semantic.items()},
        candidate_logprobs=values,semantic_logprobs={str(example.label_to_semantic[k]):values[str(k)] for k in example.labels},**audit)
    record['correct_answer_margin']=candidate_margin(record)
    return record,captured


def decode(hidden):
    return readout_value(hidden,PROBE_MEAN,PROBE_SCALE,PROBE_COEF,PROBE_INTERCEPT)


def aligned_oracle(target):
    def edit(pre,cast):
        post,audit=numerical_correction(pre,float(target),PROBE_MEAN,PROBE_SCALE,PROBE_COEF,PROBE_INTERCEPT,
            cast,tolerance=READOUT_TOLERANCE,steps=32)
        audit.update(condition='oracle',random_seed=-1,target_torque=float(target))
        return post,audit
    return edit


def aligned_random(scene_id,seed,target_norm):
    def edit(pre,cast):
        direction=deterministic_direction(pre.shape,f'20260820:{scene_id}:random:{seed}',
            np.ones_like(PROBE_SCALE),PROBE_COEF,orthogonal=False)
        post,audit=match_native_norm(pre,direction,target_norm,cast,tolerance=NORM_TOLERANCE)
        audit.update(condition='random',random_seed=int(seed),oracle_actual_norm=float(target_norm))
        return post,audit
    return edit


def capture_training_scene(examples):
    records=[];states=[]
    for example in examples:
        r,a=aligned_forward(example)
        records.append(r);states.append(a['pre'])
    native=np.stack(states).astype(np.float32)
    return records,dict(native=native,fit_mean=six_mapping_mean(native))


def capture_test_example(example):
    r,a=aligned_forward(example)
    return [r],dict(native=a['pre'])


def run_aligned_scene(example,expected_record,expected_hidden,with_controls):
    zero,a=aligned_forward(example,reference=expected_hidden,reference_input_hash=expected_record['input_fingerprint'])
    assert zero['predicted_semantic']==expected_record['predicted_semantic']
    assert max(abs(zero['candidate_logprobs'][k]-expected_record['candidate_logprobs'][k]) for k in zero['candidate_logprobs'])<=1e-6
    pre=a['pre'];fingerprint=zero['input_fingerprint']
    zero.update(condition='zero',random_seed=-1,actual_delta_norm=0.,decoded_pre=decode(pre),decoded_post=decode(pre))
    oracle,b=aligned_forward(example,aligned_oracle(example.net_torque),pre,fingerprint)
    oracle['residual']=abs(oracle['decoded_post']-float(example.net_torque))
    oracle['achieved']=oracle['residual']<=READOUT_TOLERANCE
    records=[zero,oracle];arrays=dict(pre=pre,zero_post=a['post'],oracle_post=b['post'])
    if with_controls:
        for seed in RANDOM_SEEDS:
            row,c=aligned_forward(example,aligned_random(example.scene_id,seed,oracle['actual_delta_norm']),pre,fingerprint)
            valid,error=random_validity(row['actual_delta_norm'],oracle['actual_delta_norm'],NORM_TOLERANCE)
            row.update(valid_control=valid,norm_matched=valid,norm_error_relative=error)
            records.append(row);arrays['random_'+str(seed)+'_post']=c['post']
    for r in records:
        r['readout_correct']=readout_class(r['decoded_post'],BALANCE_THRESHOLD)==r['gold_semantic']
        r['original_model_correct']=zero['correct']
        r['repaired']=not zero['correct'] and r['correct']
        r['harmed']=zero['correct'] and not r['correct']
        r['readout_version']='aligned_current_capture'
        r['probe_sha256']=PROBE_SHA
    return records,arrays


In [ ]:
%%cell_progress 8/17 Validate original fit scenes, fixed test mappings and diagnostic cohorts
try:
    from visual_torque_toolkit.behavior import BehaviorRunConfig,PromptCondition,load_behavior_examples
    old_root=PROJECT_ROOT/'results/cross_model_readout_correction_v2'
    old_model=old_root/'models'/MODEL_SLUG
    historical_probe=read_json(packet.track(old_model/'frozen_probe.json'))
    old_protocol=read_json(packet.track(old_model/'protocol.json'))
    discovery_config=read_json(packet.track(PROJECT_ROOT/'results'/MODEL_SLUG/'activation_discovery_v1/capture_config.json'))['identity']
    assert old_protocol['model_id']==MODEL_ID and old_protocol['revision']==MODEL_REVISION
    assert discovery_config['model_id']==MODEL_ID and discovery_config['resolved_revision']==MODEL_REVISION
    assert historical_probe['layer']==SELECTED_LAYER and historical_probe['alpha']==FROZEN_ALPHA
    assert old_protocol['dtype']==discovery_config['dtype']==MODEL_DTYPE
    assert old_protocol['batch_size']==1 and old_protocol['balance_threshold']==BALANCE_THRESHOLD
    assert old_protocol['readout_tolerance']==READOUT_TOLERANCE and old_protocol['norm_tolerance']==NORM_TOLERANCE
    probe_parameters(historical_probe)
    fit_manifest_path=packet.track(PROJECT_ROOT/'results/behavior_comparison_v3/activation_pilot_manifest.jsonl')
    fit_manifest=read_jsonl(fit_manifest_path)
    assert len(fit_manifest)==254 and len({str(r['scene_id']) for r in fit_manifest})==254
    assert Counter(r['split'] for r in fit_manifest)=={'probe_train':235,'probe_dev':19}
    FIT_IDS=list(map(str,historical_probe['fit_scene_ids']))
    assert len(FIT_IDS)==len(set(FIT_IDS))==254 and set(FIT_IDS)=={str(r['scene_id']) for r in fit_manifest}
    FIT_SPLITS={str(r['scene_id']):r['split'] for r in fit_manifest}
    selection=read_json(packet.track(old_root/'shared_304_manifest.json'))
    TEST_IDS=read_json(packet.track(old_root/'shared_1007_scene_ids.json'))
    assert len(TEST_IDS)==len(set(TEST_IDS))==1007 and set(TEST_IDS).isdisjoint(FIT_IDS)
    assert len(selection)==304 and len({r['scene_id'] for r in selection})==304
    selected_ids={str(r['scene_id']) for r in selection}
    CONTROL_IDS={str(r['scene_id']) for r in selection if r['orthogonal_control_subset']}
    assert len(CONTROL_IDS)==72 and selected_ids<=set(TEST_IDS)
    assert Counter(r['gold_outcome'] for r in selection)=={'left':132,'right':131,'balanced':41}
    def example_config(splits):
        return BehaviorRunConfig(dataset_dir=DATASET_DIR,output_dir=MODEL_ROOT/'unused',model_id=MODEL_ID,
            model_revision=MODEL_REVISION,conditions=(PromptCondition.IMAGE_RULE,),question_kinds=('outcome',),
            splits=tuple(splits),batch_size=1,generation_every=0,seed=20260820)
    config=example_config(('probe_train','probe_dev','probe_test'))
    assert config.dataset_fingerprint()==discovery_config['dataset_fingerprint']
    packet.track(DATASET_DIR/'scenes.jsonl');packet.track(DATASET_DIR/'splits.json')
    examples=load_behavior_examples(config)
    by_scene=defaultdict(list)
    for ex in examples:
        if str(ex.scene_id) in set(FIT_IDS)|set(TEST_IDS):by_scene[str(ex.scene_id)].append(ex)
    assert set(by_scene)==set(FIT_IDS)|set(TEST_IDS)
    for sid,group in by_scene.items():
        assert len(group)==6 and {str(e.variant) for e in group}=={'permutation_'+str(i) for i in range(6)}
        assert len({tuple(e.label_to_semantic[k] for k in e.labels) for e in group})==6
        if sid in FIT_SPLITS:assert {e.split for e in group}=={FIT_SPLITS[sid]}
        else:assert {e.split for e in group}=={'probe_test'}
    FIT_EXAMPLES={sid:sorted(by_scene[sid],key=lambda e:e.variant) for sid in FIT_IDS}
    TEST_EXAMPLES={sid:next(e for e in by_scene[sid] if e.variant==fixed_variant(sid)) for sid in sorted(TEST_IDS)}
    SELECTED_EXAMPLES=[]
    for row in selection:
        ex=TEST_EXAMPLES[str(row['scene_id'])]
        assert row['example_id']==ex.example_id and row['variant']==ex.variant
        assert row['problem_type']==ex.problem_type and row['gold_outcome']==ex.gold_semantic and int(row['net_torque'])==int(ex.net_torque)
        SELECTED_EXAMPLES.append(ex)
    used=[e for sid in FIT_IDS for e in FIT_EXAMPLES[sid]]+list(TEST_EXAMPLES.values())
    image_hashes={}
    for path in tqdm(sorted({str(e.image_path) for e in used}),desc='Checking existing images'):
        tracked=packet.track(Path(path));image_hashes[path]=packet.inputs[str(tracked)]
    INPUTS={}
    for ex in used:
        INPUTS[str(ex.example_id)]=dict(example_id=str(ex.example_id),scene_id=str(ex.scene_id),variant=str(ex.variant),
            split=str(ex.split),prompt=str(ex.prompt),labels=list(ex.labels),
            label_to_semantic={str(k):str(v) for k,v in ex.label_to_semantic.items()},
            image_sha256=image_hashes[str(ex.image_path)],net_torque=int(ex.net_torque),gold_semantic=str(ex.gold_semantic))
    assert len(INPUTS)==1524+1007
    packet.json('historical_probe_reference.json',historical_probe)
    packet.json('fit_manifest.json',fit_manifest);packet.json('diagnostic_manifest.json',selection)
    packet.json('test_fixed_mapping.json',[INPUTS[str(e.example_id)] for e in TEST_EXAMPLES.values()])
    stages.append('inputs_validated')
    print('已固定：254 个拟合场景 × 6 映射；1007 个测试场景 × 1 映射；304 个诊断场景；72 个随机对照场景。')
except Exception as error:
    record_issue('input_validation',error)


In [ ]:
%%cell_progress 9/17 Load the pinned model
if ready:
    try:
        if 'DECODER_LAYERS' in globals():
            del DECODER_LAYERS
        cell_stage(f'Loading {MODEL_KEY} weights for aligned readout validation')
        import gc
        if 'adapter' in globals():
            del adapter
            gc.collect()
            torch.cuda.empty_cache()

        adapter = HuggingFaceMultimodalAdapter(
            MODEL_ID, revision=MODEL_REVISION,
            dtype=MODEL_DTYPE, device_map=DEVICE_MAP,
        )
        display(adapter.metadata)
        assert adapter.metadata['resolved_revision'] == MODEL_REVISION

        def resolve_module_path(root, path):
            value = root
            for component in path.split('.'):
                value = getattr(value, component)
            return value

        def discover_decoder_layers(model):
            candidates = (
                'model.language_model.layers', 'model.text_model.layers',
                'language_model.model.layers', 'model.model.layers',
                'model.layers', 'language_model.layers',
            )
            for path in candidates:
                try:
                    layers = resolve_module_path(model, path)
                except AttributeError:
                    continue
                if isinstance(layers, torch.nn.ModuleList) and len(layers) >= 8:
                    return path, layers
            raise RuntimeError('Could not locate decoder layers.')

        LAYER_PATH, DECODER_LAYERS = discover_decoder_layers(adapter.model)
        assert len(DECODER_LAYERS) == EXPECTED_LAYER_COUNT
        assert all(0 <= layer < len(DECODER_LAYERS) for layer in CAPTURE_LAYERS)
        print('Layer path:', LAYER_PATH)
        print('Capture layers:', CAPTURE_LAYERS)
        assert not any(v == 'disk' for v in getattr(adapter.model,'hf_device_map',{}).values()), 'Use a GPU with enough memory for this model.'
    except Exception as error:
        record_issue('model_load',error)


In [ ]:
%%cell_progress 10/17 Freeze the current runtime and isolate its resumable captures
if ready:
    try:
        import platform,importlib.metadata,transformers
        adapter.model.eval()
        torch.backends.cuda.matmul.allow_tf32=False
        torch.backends.cudnn.benchmark=False
        assert adapter.metadata['resolved_revision']==MODEL_REVISION and LAYER_PATH==discovery_config['layer_path']
        package_versions={}
        for name in ('accelerate','torchvision','flash-attn','flash-linear-attention','causal-conv1d','safetensors'):
            try:package_versions[name]=importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:package_versions[name]=None
        runtime=dict(torch=torch.__version__,numpy=np.__version__,pandas=pd.__version__,sklearn=sklearn.__version__,
            scipy=scipy.__version__,transformers=transformers.__version__,python=platform.python_version(),cuda=torch.version.cuda,
            gpu=torch.cuda.get_device_name(0),capability=list(torch.cuda.get_device_capability(0)),
            checkpoint=MODEL_REVISION,dtype=MODEL_DTYPE,batch_size=1,layer=SELECTED_LAYER,layer_path=LAYER_PATH,
            attention_implementation=str(getattr(adapter.model.config,'_attn_implementation',None)),
            tf32_matmul=False,cudnn_benchmark=False,packages=package_versions,
            toolkit={str(p.relative_to(TOOLKIT_ROOT)):file_hash(p) for p in sorted((TOOLKIT_ROOT/'src/visual_torque_toolkit').rglob('*.py'))})
        RUN_IDENTITY=dict(protocol=PROTOCOL,runtime=runtime,input_files=dict(packet.inputs),examples_digest=digest(INPUTS))
        old_packet=packet
        RUN_ROOT=MODEL_ROOT/'runs'/digest(RUN_IDENTITY)[:20]
        packet=DiagnosticPacket(RUN_ROOT,dict(protocol=PROTOCOL,runtime_identity_sha256=digest(RUN_IDENTITY)))
        packet.inputs=dict(old_packet.inputs)
        for p in sorted(old_packet.outputs):
            if p.name!='protocol.json':packet.text(p.name,p.read_text(encoding='utf-8'))
        packet.json('runtime_identity.json',RUN_IDENTITY);packet.json('example_inputs.json',INPUTS)
        CAPTURE_IDENTITY=digest(RUN_IDENTITY)
        stages.append('runtime_locked')
        print('本次结果及断点缓存：',RUN_ROOT)
        print('不同运行身份自动进入不同目录；不读取旧实验的训练或测试激活。')
    except Exception as error:
        record_issue('runtime_identity',error)


In [ ]:
%%cell_progress 11/17 Recapture all training states with singleton forwards
if ready:
    try:
        for sid in tqdm(FIT_IDS,desc='Capturing 254 fit scenes',unit='scene'):
            group=FIT_EXAMPLES[sid]
            identity=digest([CAPTURE_IDENTITY,[INPUTS[str(e.example_id)] for e in group]])
            rows,arrays,reused=aligned_cache(RUN_ROOT,'training',sid,identity,lambda group=group:capture_training_scene(group))
            assert [r['example_id'] for r in rows]==[str(e.example_id) for e in group]
            assert arrays['native'].shape==(6,len(historical_probe['coef']))
            assert np.array_equal(arrays['fit_mean'],six_mapping_mean(arrays['native']))
            training_records.extend(rows);training_arrays.append(arrays['native'])
            training_meta.append(dict(scene_id=sid,split=FIT_SPLITS[sid],net_torque=int(group[0].net_torque)))
            CACHE_HITS+=int(reused)
        assert len(training_meta)==254 and len(training_records)==1524
        TRAIN_NATIVE=np.stack(training_arrays).astype(np.float32)
        TRAIN_MEANS=np.stack([six_mapping_mean(a) for a in TRAIN_NATIVE])
        TRAIN_TARGETS=np.asarray([r['net_torque'] for r in training_meta],float)
        TRAIN_STATE_SHA=save_arrays(packet,'training_states.npz',native=TRAIN_NATIVE,fit_means=TRAIN_MEANS,
            targets=TRAIN_TARGETS,scene_ids=np.asarray(FIT_IDS),mapping_order=np.asarray([e.variant for e in FIT_EXAMPLES[FIT_IDS[0]]]))
        packet.json('training_capture_records.json',training_records)
        packet.csv('training_scene_metadata.csv',pd.DataFrame(training_meta))
        stages.append('training_captured')
    except Exception as error:
        record_issue('training_capture',error)


In [ ]:
%%cell_progress 12/17 Refit the fixed readout using only newly captured training states
if ready:
    try:
        provenance=dict(mode='new_singleton_training_capture',capture_identity=CAPTURE_IDENTITY,
            training_states_sha256=TRAIN_STATE_SHA,training_mean_sha256=hashlib.sha256(TRAIN_MEANS.tobytes()).hexdigest(),
            layer_selection='historically_fixed_no_new_search',alpha_selection='historically_fixed_no_new_search',
            mapping_mean='sorted six variants; per-variant FP16 then FP32 mean then FP16',
            original_train_scenes=235,original_dev_scenes=19,final_refit_scenes=254,test_scenes_used=0)
        probe_path=RUN_ROOT/'aligned_probe.json'
        if probe_path.is_file():
            aligned_probe=read_json(probe_path)
            assert aligned_probe['fit_provenance']==provenance and aligned_probe['fit_scene_ids']==FIT_IDS
            assert aligned_probe['layer']==SELECTED_LAYER and aligned_probe['alpha']==FROZEN_ALPHA
            probe_parameters(aligned_probe)
        else:
            aligned_probe=fit_fixed_probe(TRAIN_MEANS,TRAIN_TARGETS,FIT_IDS,SELECTED_LAYER,FROZEN_ALPHA,provenance)
            save_json(probe_path,aligned_probe)
        packet.outputs.add(probe_path)
        PROBE_SHA=file_hash(probe_path)
        PROBE_MEAN,PROBE_SCALE,PROBE_COEF,PROBE_INTERCEPT=probe_parameters(aligned_probe)
        # Parameter fingerprints make a modified probe fail resume before inference.
        freeze(RUN_ROOT/'probe_identity.json',dict(probe_sha256=PROBE_SHA,provenance=provenance))
        packet.outputs.add(RUN_ROOT/'probe_identity.json')
        stages.append('probe_fitted')
        print('新读出器已固定；未用测试结果选层、选超参数或调阈值。')
    except Exception as error:
        record_issue('probe_fit',error)


In [ ]:
%%cell_progress 13/17 Check no-op replay and numerical writes on two fixed development examples
if ready:
    try:
        dev=sorted([e for sid in FIT_IDS if FIT_SPLITS[sid]=='probe_dev' for e in FIT_EXAMPLES[sid]],
            key=lambda e:hashlib.sha256(('aligned-write:'+str(e.example_id)).encode()).hexdigest())[:2]
        def check_development():
            rows=[]
            for ex in dev:
                clean,a=aligned_forward(ex)
                noop,b=aligned_forward(ex,lambda pre,cast:(pre.copy(),dict(condition='noop')),a['pre'],clean['input_fingerprint'])
                assert noop['actual_delta_norm']==0 and np.array_equal(a['pre'],b['post'])
                assert max(abs(clean['candidate_logprobs'][k]-noop['candidate_logprobs'][k]) for k in clean['candidate_logprobs'])<=1e-6
                oracle,c=aligned_forward(ex,aligned_oracle(ex.net_torque),a['pre'],clean['input_fingerprint'])
                random,d=aligned_forward(ex,aligned_random(ex.scene_id,0,oracle['actual_delta_norm']),a['pre'],clean['input_fingerprint'])
                valid,relative=random_validity(random['actual_delta_norm'],oracle['actual_delta_norm'],NORM_TOLERANCE)
                random.update(valid_control=valid,norm_error_relative=relative)
                rows.extend([clean,noop,oracle,random])
            return rows,dict(completed=np.asarray([len(rows)],np.int32))
        dev_checks,_,reused=aligned_cache(RUN_ROOT,'development','two_fixed_examples',
            digest([CAPTURE_IDENTITY,PROBE_SHA,[str(e.example_id) for e in dev]]),check_development)
        CACHE_HITS+=int(reused)
        packet.json('development_technical_checks.json',dev_checks)
        oracle_checks=[r for r in dev_checks if r.get('condition')=='oracle']
        assert any(r['achieved'] for r in oracle_checks), 'Neither fixed development write reached numerical tolerance; inspect checks before test interventions.'
        stages.append('development_checked')
        diagnostic_show('开发样本写入检查（不作为论文效果）',pd.DataFrame([r for r in dev_checks if 'condition' in r]),8)
    except Exception as error:
        record_issue('development_check',error)


In [ ]:
%%cell_progress 14/17 Evaluate both readouts on the same 1007 current test states
if ready:
    try:
        TEST_STATE_BY_ID={};TEST_RECORD_BY_ID={}
        for sid,ex in tqdm(list(TEST_EXAMPLES.items()),desc='Evaluating 1007 fixed test mappings',unit='scene'):
            rows,arrays,reused=aligned_cache(RUN_ROOT,'test',sid,digest([CAPTURE_IDENTITY,INPUTS[str(ex.example_id)]]),
                lambda ex=ex:capture_test_example(ex))
            assert len(rows)==1 and rows[0]['example_id']==str(ex.example_id)
            r=dict(rows[0]);h=arrays['native']
            assert h.shape==PROBE_MEAN.shape
            for name,p in [('aligned',aligned_probe),('historical',historical_probe)]:
                value=decode_with(h,p)
                r[name+'_readout']=value;r[name+'_readout_correct']=readout_class(value)==r['gold_semantic']
            test_records.append(r);test_arrays.append(h)
            TEST_STATE_BY_ID[sid]=h;TEST_RECORD_BY_ID[sid]=r;CACHE_HITS+=int(reused)
        assert len(test_records)==1007
        packet.json('test_records.json',test_records);packet.csv('test_records.csv',pd.DataFrame(test_records))
        save_arrays(packet,'test_states.npz',native=np.stack(test_arrays),scene_ids=np.asarray(list(TEST_EXAMPLES)))
        eval_summary=summarize_test(test_records)
        packet.csv('readout_answer_comparison.csv',eval_summary)
        diagnostic_show('同一当前测试状态：新旧读出器与模型答案',eval_summary[eval_summary.group.eq('all')],5)
        stages.append('test_evaluated')
    except Exception as error:
        record_issue('test_evaluation',error)


In [ ]:
%%cell_progress 15/17 Correct the true torque on 304 scenes and match random edits on the original 72
if ready:
    try:
        for ex in tqdm(SELECTED_EXAMPLES,desc='Paired correction and random controls',unit='scene'):
            sid=str(ex.scene_id)
            identity=digest([CAPTURE_IDENTITY,PROBE_SHA,INPUTS[str(ex.example_id)],sid in CONTROL_IDS])
            rows,arrays,reused=aligned_cache(RUN_ROOT,'interventions',sid,identity,
                lambda ex=ex,sid=sid:run_aligned_scene(ex,TEST_RECORD_BY_ID[sid],TEST_STATE_BY_ID[sid],sid in CONTROL_IDS))
            assert len(rows)==(6 if sid in CONTROL_IDS else 2)
            assert np.array_equal(arrays['pre'],TEST_STATE_BY_ID[sid])
            for r in rows:
                assert r['example_id']==str(ex.example_id) and r['probe_sha256']==PROBE_SHA
                key='random_'+str(r['random_seed'])+'_post' if r['condition']=='random' else r['condition']+'_post'
                post=arrays[key]
                assert abs(decode(post)-r['decoded_post'])<=1e-9
                assert abs(float(np.linalg.norm(post-arrays['pre']))-r['actual_delta_norm'])<=1e-6
                if r['condition']=='oracle':assert r['achieved']==(abs(decode(post)-ex.net_torque)<=READOUT_TOLERANCE)
                intervention_arrays.append(post);intervention_keys.append([sid,r['condition'],str(r['random_seed'])])
            intervention_records.extend(rows);CACHE_HITS+=int(reused)
        assert len(intervention_records)==896
        packet.json('intervention_records.json',intervention_records)
        packet.csv('intervention_records.csv',pd.DataFrame(intervention_records))
        save_arrays(packet,'intervention_post_states.npz',post=np.stack(intervention_arrays),keys=np.asarray(intervention_keys))
        stages.append('interventions_evaluated')
    except Exception as error:
        record_issue('interventions',error)


In [ ]:
%%cell_progress 16/17 Report numerical correction, repairs, harms, matched controls and paired intervals
if ready:
    try:
        effect_summary,contrasts,technical=summarize_interventions(intervention_records,
            [str(e.scene_id) for e in SELECTED_EXAMPLES],CONTROL_IDS)
        packet.csv('correction_summary.csv',effect_summary);packet.csv('paired_effects.csv',contrasts)
        f=pd.DataFrame(intervention_records)
        packet.csv('oracle_write_failures.csv',f[f.condition.eq('oracle')&f.achieved.ne(True)])
        packet.csv('random_norm_failures.csv',f[f.condition.eq('random')&f.valid_control.ne(True)])
        packet.json('technical_summary.json',technical)
        diagnostic_show('修准是否成功，以及修好／改错多少题',effect_summary,5)
        diagnostic_show('相同场景的准确率和正确答案得分比较',contrasts,10)
        print('所有尝试保留；修准成功子集和四个随机对照均有效的共同子集另列。')
        print('区间不足以排除较大改善时，不能把未检出提升解释为没有作用。')
        stages.append('analysis_complete')
    except Exception as error:
        record_issue('analysis',error)


In [ ]:
%%cell_progress 17/17 Download the complete result packet or an explicitly incomplete packet
changed_inputs=[p for p,h in packet.inputs.items() if not Path(p).is_file() or file_hash(p)!=h]
if changed_inputs:
    record_issue('source_verification',RuntimeError('Input files changed during this run: '+repr(changed_inputs)))
expected_stages={'inputs_validated','runtime_locked','training_captured','probe_fitted','development_checked',
    'test_evaluated','interventions_evaluated','analysis_complete'}
complete=bool(ready and set(stages)==expected_stages and len(training_meta)==254 and len(test_records)==1007 and len(intervention_records)==896)
summary=dict(status='complete' if complete else 'partial',workflow_complete=complete,model=MODEL_KEY,
    fit_scenes_completed=len(training_meta),fit_mapping_records=len(training_records),test_scenes_completed=len(test_records),
    intervention_records_completed=len(intervention_records),new_forward_attempts=NEW_FORWARD_COUNT,cache_groups_reused=CACHE_HITS,
    stages=stages,issues=issues,technical=technical,historical_activation_reuse=False,historical_test_cache_reuse=False,
    fixed_layer=SELECTED_LAYER,fixed_alpha=FROZEN_ALPHA,batch_size=1,new_independent_confirmation=False,
    primary_mapping='fixed single mapping; original six-map average retained only for fit',original_scientific_decisions_unchanged=True)
packet.json('progress.json',summary)
# A failed stage still exports completed rows, not stale outputs from an older invocation.
if not complete:
    packet.json('partial_training_records.json',training_records)
    packet.json('partial_test_records.json',test_records)
    packet.json('partial_intervention_records.json',intervention_records)
    if training_arrays:
        save_arrays(packet,'partial_training_states.npz',native=np.stack(training_arrays),
            scene_ids=np.asarray([r['scene_id'] for r in training_meta]))
    if test_arrays:
        save_arrays(packet,'partial_test_states.npz',native=np.stack(test_arrays),
            scene_ids=np.asarray([r['scene_id'] for r in test_records]))
    if intervention_arrays:
        save_arrays(packet,'partial_intervention_post_states.npz',post=np.stack(intervention_arrays),
            keys=np.asarray(intervention_keys))
lines=['# 统一捕获条件的读出验证',f'模型：{MODEL_ID}；层位：{SELECTED_LAYER}；ridge alpha：{FROZEN_ALPHA}。',
    '训练、测试和干预均采用当前逐张 BF16 前向。只重新拟合外部读出器，大模型权重不变。',
    '原 254 个场景用于固定配置的最终拟合；训练保留六映射平均，主要测试和干预使用原来固定的一种映射。',
    '原测试场景已经被查看，本轮属于协议稳健性复核，不是新的独立盲测。']
if complete:
    main=effect_summary[effect_summary.group.eq('all_304')].iloc[0]
    aligned=eval_summary[eval_summary.group.eq('all')&eval_summary.probe.eq('aligned')].iloc[0]
    lines += [f'1007 个测试场景中，新读出器判断正确 {int(aligned.readout_correct)} 个，模型答对 {int(aligned.model_correct)} 个；读出正确但模型答错 {int(aligned.readout_correct_model_wrong)} 个。',
        f'304 个诊断场景中，数值修准达标 {int(main.oracle_achieved)} 个；答对数量 {int(main.before_correct)} → {int(main.after_correct)}；修好 {int(main.repaired)} 个原错误，新增 {int(main.harmed)} 个错误。',
        f'准确率变化 {main.accuracy_change*100:.2f} 个百分点，95% 场景配对区间 [{main.lower*100:.2f}, {main.upper*100:.2f}]。',
        f'随机对照实际幅度未达标 {technical["random_invalid"]}/{technical["random_attempted"]} 次；修准达标且四个随机方向均有效的共同子集有 {technical["common_valid_control_scenes"]} 个场景。',
        '完整总体、平衡题、非平衡题、有效子集和随机对照比较分别见 CSV；未通过技术检查的尝试没有被删除。',
        '同一当前状态上的历史读出器数值仅作为参照；不将旧运行的修正效果混入本轮成对比较。']
else:
    lines+=['本次未完整完成；只能使用明确完成的部分，不能给出完整验证结论。',json.dumps(issues,ensure_ascii=False,indent=2)]
lines+=['数值修准使用真实净力矩；它检验所选外部读出器方向的作用，不自动证明模型原生物理计算被修复。',
    '原 5 个百分点实质改善标准仅用于解释区间，不能把不显著当作零作用；技术失败和有效子集必须同时查看。',
    '原有 Qwen-27B、微调、内容确认及修正—阻断结果均未修改。']
packet.finish(summary,'\n\n'.join(lines)+'\n','VisualTorque_Aligned_Readout_'+MODEL_KEY+'_Output.zip',AUTO_DOWNLOAD)
